In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [2]:
eeg_channels = [
    "FP1","FP2","F7","F3",
    "FZ","F4","F8","C4"
]

In [3]:
def create_windows(df,task_type,window_size=250):

    X = []
    y = []
    groups = []

    scaler = StandardScaler()

    df[eeg_channels] = scaler.fit_transform(
        df[eeg_channels]
    )

    for start in range(0,len(df) - window_size,window_size):

        window = df.iloc[
            start:start+window_size
        ]

        X.append(
            window[eeg_channels].values
        )

        y.append(
            window["Stress_Level"].iloc[0]
        )

        groups.append(
            f"{task_type}_"
            f"{window['Subject_ID'].iloc[0]}"
        )

    return X, y, groups

In [4]:
all_X = []
all_y = []
all_groups = []

tasks = [
    "Arithmetic",
    "Stroop"
]

for task in tasks:

    base_dir = f"Dataset/Clean/{task}"

    for category in os.listdir(base_dir):

        category_path = os.path.join(
            base_dir,
            category
        )

        if not os.path.isdir(category_path):
            continue

        for file in os.listdir(category_path):

            if not file.endswith(".csv"):
                continue

            filepath = os.path.join(category_path,file)

            df = pd.read_csv(filepath)

            X_part, y_part, group_part = (
                create_windows(df,task_type=task,window_size=250)
            )

            all_X.extend(X_part)
            all_y.extend(y_part)
            all_groups.extend(group_part)

In [5]:
X = np.array(all_X)

y = np.array(all_y)

groups = np.array(all_groups)

print(X.shape)
print(y.shape)
print(groups.shape)

(11036, 250, 8)
(11036,)
(11036,)


In [6]:
from sklearn.preprocessing import LabelEncoder
import os

base_dir = "Dataset/Train/DL"
os.makedirs(base_dir, exist_ok=True)

X_filepath = os.path.join(base_dir,"X_windows.npy")
y_filepath = os.path.join(base_dir,"y_windows.npy")
groups_filepath = os.path.join(base_dir,"groups.npy")
y_encoded_filepath = os.path.join(base_dir,"y_encoded.npy")

le = LabelEncoder()
y_encoded = le.fit_transform(y)

np.save(X_filepath, X)
np.save(y_filepath, y)
np.save(groups_filepath, groups)
np.save(y_encoded_filepath, y_encoded)

In [7]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print(label_encoder.classes_)

['highlevel' 'lowlevel' 'midlevel' 'natural']


In [8]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        X,
        y_encoded,
        groups
    )
)

X_train = X[train_idx]
X_test = X[test_idx]

y_train = y_encoded[train_idx]
y_test = y_encoded[test_idx]

print(X_train.shape)
print(X_test.shape)

(8856, 250, 8)
(2180, 250, 8)


In [9]:
from tensorflow.keras.utils import to_categorical

num_classes = 4

y_train_cat = to_categorical(
    y_train,
    num_classes
)

y_test_cat = to_categorical(
    y_test,
    num_classes
)

In [10]:
from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    Flatten,
    Dense,
    Dropout
)

cnn_model = Sequential([

    Conv1D(
        filters=64,
        kernel_size=3,
        activation="relu",
        input_shape=(250,8)
    ),

    MaxPooling1D(2),

    Conv1D(
        filters=128,
        kernel_size=3,
        activation="relu"
    ),

    MaxPooling1D(2),

    Flatten(),

    Dense(
        128,
        activation="relu"
    ),

    Dropout(0.3),

    Dense(
        4,
        activation="softmax"
    )
])

cnn_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

cnn_model.summary()

d:\VIT_Projects\cognitive\cogenv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 248, 64)        │         1,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 124, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 122, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 61, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 7808)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       999,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,026,372 (3.92 MB)

 Trainable params: 1,026,372 (3.92 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
history = cnn_model.fit(

    X_train,
    y_train_cat,

    validation_data=(
        X_test,
        y_test_cat
    ),

    epochs=20,
    batch_size=64
)

Epoch 1/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.5346 - loss: 1.0724 - val_accuracy: 0.3615 - val_loss: 1.6726
Epoch 2/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6973 - loss: 0.7355 - val_accuracy: 0.3807 - val_loss: 1.9473
Epoch 3/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.7800 - loss: 0.5525 - val_accuracy: 0.4046 - val_loss: 2.2483
Epoch 4/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8337 - loss: 0.4418 - val_accuracy: 0.4032 - val_loss: 2.3383
Epoch 5/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8666 - loss: 0.3604 - val_accuracy: 0.4321 - val_loss: 2.5045
Epoch 6/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.8941 - loss: 0.2912 - val_accuracy: 0.4069 - val_loss: 3.0357
Epoch 7/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.9068 - loss: 0.2533 - val_accuracy: 0.4349 - val_loss: 3.0428
Epoch 8/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.9239 - loss: 0.2154 - val_accu

In [12]:
loss, accuracy = cnn_model.evaluate(
    X_test,
    y_test_cat
)

print(
    "CNN Accuracy:",
    accuracy
)

69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4381 - loss: 5.3681
CNN Accuracy: 0.43807339668273926


In [13]:
from tensorflow.keras.layers import LSTM

lstm_model = Sequential([

    LSTM(
        64,
        input_shape=(250,8)
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dropout(0.3),

    Dense(
        4,
        activation="softmax"
    )
])

lstm_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

d:\VIT_Projects\cognitive\cogenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [14]:
history = lstm_model.fit(

    X_train,
    y_train_cat,

    validation_data=(
        X_test,
        y_test_cat
    ),

    epochs=20,
    batch_size=64
)

Epoch 1/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 10s 67ms/step - accuracy: 0.4474 - loss: 1.2248 - val_accuracy: 0.3211 - val_loss: 1.4687
Epoch 2/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 9s 63ms/step - accuracy: 0.5553 - loss: 1.0578 - val_accuracy: 0.3573 - val_loss: 1.5013
Epoch 3/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 9s 65ms/step - accuracy: 0.5732 - loss: 1.0052 - val_accuracy: 0.3450 - val_loss: 1.5499
Epoch 4/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 18s 128ms/step - accuracy: 0.6047 - loss: 0.9406 - val_accuracy: 0.3665 - val_loss: 1.5867
Epoch 5/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 10s 68ms/step - accuracy: 0.6378 - loss: 0.8785 - val_accuracy: 0.3830 - val_loss: 1.6169
Epoch 6/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 7s 54ms/step - accuracy: 0.6652 - loss: 0.8039 - val_accuracy: 0.3711 - val_loss: 1.6654
Epoch 7/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 8s 56ms/step - accuracy: 0.6514 - loss: 0.8544 - val_accuracy: 0.3633 - val_loss: 1.7257
Epoch 8/20
139/139 ━━━━━━━━━━━━━━━━━━━━ 8s 56ms/step - accuracy: 0.6808 - loss: 0.7719 - val_

In [15]:
loss, accuracy = lstm_model.evaluate(
    X_test,
    y_test_cat
)

print(
    "LSTM Accuracy:",
    accuracy
)

69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.3995 - loss: 2.2236
LSTM Accuracy: 0.39954128861427307


In [16]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import pandas as pd
import numpy as np

# CNN Predictions
cnn_pred = np.argmax(
    cnn_model.predict(X_test),
    axis=1
)

# LSTM Predictions
lstm_pred = np.argmax(
    lstm_model.predict(X_test),
    axis=1
)

# True Labels
y_true = np.argmax(
    y_test_cat,
    axis=1
)

cnn_results = {
    "Model": "CNN",
    "Accuracy": accuracy_score(y_true, cnn_pred),
    "Precision": precision_score(
        y_true,
        cnn_pred,
        average="weighted"
    ),
    "Recall": recall_score(
        y_true,
        cnn_pred,
        average="weighted"
    ),
    "F1 Score": f1_score(
        y_true,
        cnn_pred,
        average="weighted"
    )
}

lstm_results = {
    "Model": "LSTM",
    "Accuracy": accuracy_score(y_true, lstm_pred),
    "Precision": precision_score(
        y_true,
        lstm_pred,
        average="weighted"
    ),
    "Recall": recall_score(
        y_true,
        lstm_pred,
        average="weighted"
    ),
    "F1 Score": f1_score(
        y_true,
        lstm_pred,
        average="weighted"
    )
}

comparison_dl = pd.DataFrame([
    cnn_results,
    lstm_results
])

comparison_dl

69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step


,Model,Accuracy,Precision,Recall,F1 Score
0,CNN,0.438073,0.440280,0.438073,0.438517
1,LSTM,0.399541,0.402385,0.399541,0.397014
